# ECG-FM Stage 2 — Colab

**Before running:** Runtime → Change runtime type → **T4 GPU**

### Session workflow
- **First time only:** Run cells 1 → 2 → 3 → 4 (downloads PTB-XL to Drive, ~3 hrs)
- **Every session (incl. after reconnect):** Run cells 1 → 2 → **5** → 6 → 7
- Cell 4 uses `wget -nc` (no-clobber) — safe to re-run, skips files already on Drive

In [ ]:
# Cell 1 — Check GPU
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU — Runtime → Change runtime type → T4 GPU')
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

import psutil
print('RAM :', round(psutil.virtual_memory().total / 1e9, 1), 'GB')

In [ ]:
# Cell 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
MODEL_DIR = '/content/drive/MyDrive/EKG_models'
os.makedirs(MODEL_DIR, exist_ok=True)

for f in ['mimic_iv_ecg_physionet_pretrained.pt', 'ecgfm_stage1.pt']:
    status = 'OK' if os.path.exists(f'{MODEL_DIR}/{f}') else 'MISSING'
    print(f'  {status} — {f}')

In [ ]:
# Cell 3 — Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'wfdb', 'scipy', 'scikit-learn'], check=True)
print('Dependencies ready')

In [ ]:
# Cell 4 — Download PTB-XL 500Hz to Drive (FIRST TIME ONLY — takes ~3 hrs)
# Safe to re-run: wget -nc skips files already on Drive.
# If disconnected mid-download: reconnect, re-run cells 1-2, then re-run this cell.

import os
from pathlib import Path

MODEL_DIR    = '/content/drive/MyDrive/EKG_models'
DRIVE_PTBXL  = f'{MODEL_DIR}/ptbxl'
os.makedirs(DRIVE_PTBXL, exist_ok=True)

for fname in ['ptbxl_database.csv', 'scp_statements.csv']:
    if not os.path.exists(f'{DRIVE_PTBXL}/{fname}'):
        os.system(f'wget -q -nc -P {DRIVE_PTBXL} https://physionet.org/files/ptb-xl/1.0.3/{fname}')
    print(f'  {"OK" if os.path.exists(f"{DRIVE_PTBXL}/{fname}") else "MISSING"} — {fname}')

rec_dir = Path(f'{DRIVE_PTBXL}/records500')
n_dat = len(list(rec_dir.rglob('*.dat'))) if rec_dir.exists() else 0
print(f'\nSignal files on Drive: {n_dat} / 21837')

if n_dat < 21000:
    print('Downloading records500 to Drive (skipping files already present)...')
    os.system(f'wget -r -nc -np -nH --cut-dirs=3 -P {DRIVE_PTBXL} https://physionet.org/files/ptb-xl/1.0.3/records500/')
    n_dat = len(list(rec_dir.rglob('*.dat')))
    print(f'After download: {n_dat} .dat files on Drive')
else:
    print('All records already on Drive — skipping download')

print('\nCell 4 done. Run Cell 5 to set up the session, then Cell 6 to train.')

In [ ]:
# Cell 5 — Session setup (run this EVERY session, including after reconnect)
# Sets up: training scripts, model weights, PTB-XL symlink, working directory

import os, shutil
os.chdir('/content')

MODEL_DIR   = '/content/drive/MyDrive/EKG_models'
DRIVE_PTBXL = f'{MODEL_DIR}/ptbxl'

# ── 1. Training scripts from Kaggle ──────────────────────────────────────────
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_3ee6b0fec65eecf1454734b63b5dc062'
import kagglehub
src = kagglehub.dataset_download('shlomihazan/ecgfm-models')
for f in ['ecgfm_finetune.py', 'ecgfm_encoder.py', 'cnn_classifier.py']:
    shutil.copy(f'{src}/{f}', f'/content/{f}')

# ── 2. Model weights ─────────────────────────────────────────────────────────
os.makedirs('/content/models/ecgfm', exist_ok=True)
shutil.copy(f'{MODEL_DIR}/mimic_iv_ecg_physionet_pretrained.pt',
            '/content/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt')
shutil.copy(f'{MODEL_DIR}/ecgfm_stage1.pt', '/content/models/ecgfm_stage1.pt')

# ── 3. PTB-XL symlink ────────────────────────────────────────────────────────
os.makedirs('/content/ekg_datasets', exist_ok=True)
link = '/content/ekg_datasets/ptbxl'
if os.path.islink(link) or os.path.exists(link):
    if os.path.islink(link): os.remove(link)
    else: shutil.rmtree(link)
os.symlink(DRIVE_PTBXL, link)

# ── 4. Verify ─────────────────────────────────────────────────────────────────
checks = {
    'ecgfm_finetune.py':                      os.path.exists('/content/ecgfm_finetune.py'),
    'models/ecgfm_stage1.pt':                 os.path.exists('/content/models/ecgfm_stage1.pt'),
    'models/ecgfm/pretrained encoder':        os.path.exists('/content/models/ecgfm/mimic_iv_ecg_physionet_pretrained.pt'),
    'ekg_datasets/ptbxl/ptbxl_database.csv':  os.path.exists('/content/ekg_datasets/ptbxl/ptbxl_database.csv'),
    'signal files (sample)':                  os.path.exists('/content/ekg_datasets/ptbxl/records500/00000/00001_hr.dat'),
}
all_ok = True
for k, v in checks.items():
    print(f'  {"OK" if v else "MISSING"} — {k}')
    if not v: all_ok = False

print()
if all_ok:
    print('All OK — run Cell 6 to start training')
else:
    print('Fix MISSING items before running Cell 6')

In [ ]:
# Cell 6 — Keep-alive (run before Cell 7 to prevent idle disconnect)
import time, threading

def _keep_alive():
    i = 0
    while True:
        time.sleep(30)
        i += 1
        print(f'  [keep-alive {i}]', flush=True)

t = threading.Thread(target=_keep_alive, daemon=True)
t.start()
print('Keep-alive started (prints every 30s)')

In [ ]:
# Cell 7 — Train Stage 2
# ~20 min/epoch on T4, ~3 hrs total. Saves models/ecgfm_stage2.pt on each improvement.
# OOM? change --batch_s2 8 to --batch_s2 4

import os
os.chdir('/content')
!python -u ecgfm_finetune.py --stage 2 --batch_s2 8

In [ ]:
# Cell 8 — Save result to Drive
import os, shutil, torch

MODEL_DIR = '/content/drive/MyDrive/EKG_models'

if os.path.exists('/content/models/ecgfm_stage2.pt'):
    dest = f'{MODEL_DIR}/ecgfm_stage2.pt'
    shutil.copy('/content/models/ecgfm_stage2.pt', dest)
    print(f'Saved to Drive ({round(os.path.getsize(dest)/1e6)} MB)')

    ckpt = torch.load('/content/models/ecgfm_stage2.pt', map_location='cpu', weights_only=False)
    tm   = ckpt.get('test_metrics', {})
    print(f'\nFinal test results:')
    print(f'  Accuracy  : {tm.get("acc", 0):.1%}')
    print(f'  HYP F1    : {tm.get("hyp_f1", 0):.3f}  (baseline 0.375)')
    print(f'  Macro F1  : {tm.get("macro_f1", 0):.3f}  (baseline 0.492)')
else:
    print('WARNING: ecgfm_stage2.pt not found')